
### Elasticsearch

<img src="https://static-www.elastic.co/v3/assets/bltefdd0b53724fa2ce/blt5ebe80fb665aef6b/5ea8c8f26b62d4563b6ecec2/brand-elasticsearch-220x130.svg" width="300">

This notebook has been adapted to use the Python library rather than installing from Maven.  This is simpler but will suffer at scale because it is not distributed.

Instructions for the Spark driver:
- [Databricks docs](https://docs.databricks.com/aws/en/archive/connectors/elasticsearch)
- [Driver Repo]()

1. Launch a cluster in your workspace or choose an existing cluster.
2. Once the new cluster is running, go to the "Libraries" tab of that cluster, and click "Install new" -> choose "Maven" -> enter the maven coordinates `org.elasticsearch:elasticsearch-spark-30_2.13:8.4.3` -> click "Install". If running into errors like `org.elasticsearch.hadoop.EsHadoopIllegalArgumentException: Cannot detect ES version` while the ES connection is verified, consider install newer versions that matches your ES service.
3. Once the installation has finished, attach this notebook to the cluster, and run write and/or read operations against your Elasticsearch cluster

I ran into multiple issues trying to get the right version of the driver.

Instructions for the Python driver:
1. Just pip install into a Serverless cluster

In [0]:
%pip install elasticsearch --quiet

## Secrets
Store sensitive information in secrets.  From the command line:

`databricks secrets create-scope elasticsearch`

`databricks secrets put-secret elasticsearch serverless_api_key`

`databricks secrets put-secret elasticsearch username`

`databricks secrets put-secret elasticsearch password`

In [0]:
serverless_hostname = "my-elasticsearch-project-c1579c.es.us-east-1.aws.elastic.cloud" # serverless
hosted_hostname = "mytestes-0b283a.es.us-east-1.aws.found.io" # managed
port = "443"
ssl = "true"
index = "people"

# Read secrets
username = dbutils.secrets.get("elasticsearch", "username")
password = dbutils.secrets.get("elasticsearch", "password")
serverless_api_key = dbutils.secrets.get("elasticsearch", "serverless_api_key")

In [0]:
people = spark.createDataFrame( [ ("Bilbo",     50), 
                                  ("Gandalf", 1000), 
                                  ("Thorin",   195),  
                                  ("Balin",    178), 
                                  ("Kili",      77),
                                  ("Dwalin",   169), 
                                  ("Oin",      167), 
                                  ("Gloin",    158), 
                                  ("Fili",      82), 
                                  ("Bombur",  None)
                                ], 
                                ["name", "age"] 
                              )

In [0]:
# Convert Spark DataFrame to list of dicts and bulk index
import json
rows = json.loads(people.toPandas().to_json(orient="records"))
actions = [{"_index": index, "_source": row} for row in rows]

## Serverless

In [0]:
import subprocess
result = subprocess.run(["nc", "-vz", serverless_hostname, port], capture_output=True, text=True)
print(result.stdout)
print(result.stderr)

In [0]:
from elasticsearch import Elasticsearch
from elasticsearch.helpers import bulk

serverless_client = Elasticsearch(
        f"https://{serverless_hostname}:{port}",
        api_key=serverless_api_key
)

# Delete existing index for overwrite behavior
if serverless_client.indices.exists(index=index):
    serverless_client.indices.delete(index=index)


success, errors = bulk(serverless_client, actions)
print(f"Successfully indexed {success} documents to '{index}'")
if errors:
    print(f"Errors: {errors}")

In [0]:
# Read all documents from the index using the Python client
result = serverless_client.search(index=index, query={"match_all": {}}, size=1000)
hits = [hit["_source"] for hit in result["hits"]["hits"]]

# Convert to Spark DataFrame
df = spark.createDataFrame(hits)
display(df)

## Hosted

In [0]:
import subprocess
result = subprocess.run(["nc", "-vz", hosted_hostname, port], capture_output=True, text=True)
print(result.stdout)
print(result.stderr)

In [0]:
from elasticsearch import Elasticsearch
from elasticsearch.helpers import bulk

# Connect using basic auth over HTTPS
es = Elasticsearch(
    f"https://{hosted_hostname}:{port}",
    basic_auth=(username, password),
    verify_certs=True
)

# Delete existing index for overwrite behavior
if es.indices.exists(index=index):
    es.indices.delete(index=index)



success, errors = bulk(es, actions)
print(f"Successfully indexed {success} documents to '{index}'")
if errors:
    print(f"Errors: {errors}")

In [0]:
# Read all documents from the index using the Python client
result = es.search(index=index, query={"match_all": {}}, size=1000)
hits = [hit["_source"] for hit in result["hits"]["hits"]]

# Convert to Spark DataFrame
df = spark.createDataFrame(hits)
display(df)

## Save in Delta 

In [0]:
# Creates a Delta table called table_name
df.write.format("delta").saveAsTable(table_name)